# PUMA Colab Training (Project Scripts + W&B)

Notebook nay tao moi cho Colab.

Flow:
1. Clone repo + cai dependencies.
2. Mount Google Drive (dataset + checkpoint).
3. Convert dataset_PUMA ve format train cua project.
4. Tao config train dung input size 1024 va dung so class theo track.
5. Chay `scripts/train.py` va theo doi bang Weights & Biases.

In [ ]:
# ===== 1) Clone project repo =====
# Ban co the doi REPO_URL theo fork cua ban.
REPO_URL = 'https://github.com/hoangtung386/PUMA_Nucleus_Segmentation.git'
REPO_DIR = 'PUMA_Nucleus_Segmentation'

!rm -rf {REPO_DIR}
!git clone {REPO_URL}
%cd {REPO_DIR}
!ls -la

In [ ]:
# ===== 2) Install dependencies =====
!pip -q install -U pip
!pip -q install -e .[dev,cellpose]
!pip -q install requirements.txt

In [ ]:
# ===== 3) Mount Drive and define paths =====
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
REPO_ROOT = Path('/content') / REPO_DIR

# Dataset root trong Drive (giu nguyen cau truc dataset_PUMA)
DATASET_ROOT = Path('/content/drive/MyDrive/dataset_PUMA')

RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
SPLIT_DIR = REPO_ROOT / 'data' / 'splits'

# Noi luu train outputs/checkpoint tren Drive
DRIVE_RUN_ROOT = Path('/content/drive/MyDrive/PUMA_experiments')
CKPT_ROOT = DRIVE_RUN_ROOT / 'models'
LOG_ROOT = DRIVE_RUN_ROOT / 'runs'
OUT_ROOT = DRIVE_RUN_ROOT / 'outputs'

for p in [RAW_DIR / 'images', RAW_DIR / 'annotations', PROCESSED_DIR, SPLIT_DIR, CKPT_ROOT, LOG_ROOT, OUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('CKPT_ROOT:', CKPT_ROOT)

In [ ]:
# ===== 4) Select track =====
TRACK = 1  # 1 or 2 (class count set in config)

from puma_seg.data.geojson_parser import NUM_CLASSES
n_fg = NUM_CLASSES[TRACK] - 1
assert (TRACK == 1 and n_fg == 3) or (TRACK == 2 and n_fg == 10)
print(f'Track {TRACK} -> {n_fg} foreground classes (config loaded in cell 7)')

In [ ]:
# ===== 5) Build raw dataset expected by scripts/prepare_data.py =====
# Expected layout:
#   data/raw/images/<stem>.tif
#   data/raw/annotations/<stem>.geojson
# Source nuclei anno in dataset_PUMA: <stem>_nuclei.geojson

import shutil

img_src_dir = DATASET_ROOT / '01_training_dataset_tif_ROIs'
ann_src_dir = DATASET_ROOT / '01_training_dataset_geojson_nuclei'
assert img_src_dir.exists(), f'Missing {img_src_dir}'
assert ann_src_dir.exists(), f'Missing {ann_src_dir}'

img_dst_dir = RAW_DIR / 'images'
ann_dst_dir = RAW_DIR / 'annotations'

for p in img_dst_dir.glob('*'):
    p.unlink()
for p in ann_dst_dir.glob('*'):
    p.unlink()

copied, missing = 0, []
for img_path in sorted(img_src_dir.glob('*.tif')):
    stem = img_path.stem
    ann_path = ann_src_dir / f'{stem}_nuclei.geojson'
    if not ann_path.exists():
        missing.append(stem)
        continue
    shutil.copy2(img_path, img_dst_dir / f'{stem}.tif')
    shutil.copy2(ann_path, ann_dst_dir / f'{stem}.geojson')
    copied += 1

print('Copied pairs:', copied)
print('Missing annotations:', len(missing))
print('Images:', len(list(img_dst_dir.glob('*.tif'))))
print('Annotations:', len(list(ann_dst_dir.glob('*.geojson'))))

In [ ]:
# ===== 6) Prepare processed data =====
import yaml
import sys

# Add project to path for direct imports
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'scripts'))

from puma_seg.data.geojson_parser import parse_geojson
from puma_seg.utils.io_utils import list_image_paths
from prepare_data import make_split, process_one

# Load config for data split parameters
config_path = REPO_ROOT / 'configs' / 'multitask.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    base_cfg = yaml.safe_load(f)

SEED = 42
img_dir = RAW_DIR / 'images'
ann_dir = RAW_DIR / 'annotations'

# List and split images
image_paths = list_image_paths(img_dir, extensions=('.tif',))
stems = [p.stem for p in image_paths]
split = make_split(stems, base_cfg['data']['val_split'], base_cfg['data']['test_split'], SEED)

# Save split file
split_path = SPLIT_DIR / 'split.json'
import json
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
with open(split_path, 'w', encoding='utf-8') as f:
    json.dump(split, f, indent=2)
print(f'Split: {len(split["train"])} train / {len(split["val"])} val / {len(split["test"])} test')

# Process each split
from tqdm import tqdm
stem_to_path = {p.stem: p for p in image_paths}
for subset in ('train', 'val', 'test'):
    out_subset = PROCESSED_DIR / subset
    print(f'Processing {subset}...')
    for stem in tqdm(split[subset]):
        img_path = stem_to_path.get(stem)
        if img_path:
            process_one(img_path, ann_dir, out_subset, TRACK, '.tif')
print('Done preparing data.')

In [ ]:
# ===== 7) Login W&B and init run =====
import os
import yaml
from datetime import datetime
import wandb

# Use Colab userdata for WANDB_TOKEN (more secure than environment variable)
from google.colab import userdata
WANDB_TOKEN = userdata.get('WANDB_TOKEN')
wandb.login(key=WANDB_TOKEN)

run_name = f'track{TRACK}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

In [ ]:
# Load base config from configs/ folder
config_path = REPO_ROOT / 'configs' / 'multitask.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# Override with Colab-specific values
cfg['data']['raw_dir'] = str(RAW_DIR)
cfg['data']['processed_dir'] = str(PROCESSED_DIR)
cfg['data']['split_file'] = str(SPLIT_DIR / 'split.json')
cfg['data']['track'] = TRACK
cfg['data']['image_size'] = None  # keep original 1024x1024
cfg['data']['val_split'] = VAL_SPLIT
cfg['data']['test_split'] = TEST_SPLIT

cfg['segmentation']['n_epochs'] = SEG_EPOCHS
cfg['segmentation']['batch_size'] = SEG_BATCH_SIZE
cfg['segmentation']['gradient_accumulation_steps'] = SEG_GRAD_ACCUM
cfg['segmentation']['model_name'] = f'puma_cellpose_{run_name}'

cfg['classification']['phase1_epochs'] = CLS_PHASE1_EPOCHS
cfg['classification']['phase2_epochs'] = CLS_PHASE2_EPOCHS
cfg['classification']['batch_size'] = CLS_BATCH_SIZE
cfg['classification']['gradient_accumulation_steps'] = CLS_GRAD_ACCUM
cfg['classification']['patience'] = 8
cfg['classification']['model_name'] = f'best_classifier_{run_name}'

cfg['paths']['save_dir'] = str(CKPT_ROOT / run_name)
cfg['paths']['log_dir'] = str(LOG_ROOT / run_name)
cfg['paths']['output_dir'] = str(OUT_ROOT / run_name)

cfg_path = REPO_ROOT / 'configs' / f'colab_{run_name}.yaml'
with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

wandb_run = wandb.init(
    project='puma-nucleus-segmentation',
    name=run_name,
    config=cfg,
    job_type='train',
)
wandb.save(str(cfg_path))
print('Config path:', cfg_path)

In [ ]:
# ===== 8) Train using project functions =====
import sys
import argparse

# Add project to path for direct imports
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'scripts'))

from puma_seg.cli._train_impl import load_config, run_segmentation, run_classification

# Load config and override paths
cfg = load_config(cfg_path)

# Run segmentation training
print('Training segmentation...')
args = argparse.Namespace(
    config=cfg_path,
    mode='segmentation',
    resume_seg=None,
    resume_cls=None,
    dry_run=False,
)
run_segmentation(cfg, args)

# Run classification training
print('Training classification...')
args = argparse.Namespace(
    config=cfg_path,
    mode='classification',
    resume_seg=None,
    resume_cls=None,
    dry_run=False,
)
run_classification(cfg, args)

print('Training done.')

In [ ]:
# ===== 10) Evaluate metrics =====
import sys
import numpy as np

sys.path.insert(0, str(REPO_ROOT / 'src'))

from puma_seg.evaluation.metrics import evaluate_predictions, compute_puma_score
from puma_seg.data.geojson_parser import CLASS_NAMES_T1, CLASS_NAMES_T2
from puma_seg.data.dataset import PUMASegmentationDataset

# Load test dataset
test_ds = PUMASegmentationDataset(
    processed_dir=PROCESSED_DIR,
    split='test',
    track=TRACK,
)

# Get class names
class_names = CLASS_NAMES_T1 if TRACK == 1 else CLASS_NAMES_T2
n_fg = len(class_names) - 1  # exclude background

print(f'Test set: {len(test_ds)} images')
print(f'Classes: {class_names[1:]}')

# Note: For full evaluation, need to run predictions on test set first
# This cell computes metrics from saved predictions
print('Evaluating... (run prediction first for full metrics)')

# placeholder results for demo
wandb.log({'test_puma_score': 0.0, 'test_micro_f1': 0.0})

In [ ]:
# ===== 11) Visualize predictions =====
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(REPO_ROOT / 'src'))

from puma_seg.utils.visualization import plot_sample, plot_predictions_vs_gt, overlay_instances
from puma_seg.data.dataset import PUMASegmentationDataset, PUMASegmentationDataset

# Load sample from test set
test_ds = PUMASegmentationDataset(
    processed_dir=PROCESSED_DIR,
    split='test',
    track=TRACK,
)

# Visualize first 3 test samples
for i in range(min(3, len(test_ds))):
    image, instance_mask = test_ds[i]
    
    # Load class mask if available
    labels_path = PROCESSED_DIR / 'test' / 'labels' / f'{test_ds.image_stems[i]}.npy'
    if labels_path.exists():
        class_mask = np.load(labels_path)
    else:
        class_mask = None
    
    plot_sample(image, instance_mask, class_mask, track=TRACK, title=f'Test sample {i}')
    plt.show()

print('Visualization done.')
wandb.log({'visualization_sample': plt.gcf()})

In [ ]:
# ===== 12) Log checkpoints as W&B artifact =====
from pathlib import Path

save_dir = Path(cfg['paths']['save_dir'])
log_dir = Path(cfg['paths']['log_dir'])

print('save_dir:', save_dir)
print('log_dir:', log_dir)

artifact = wandb.Artifact(name=f'puma-checkpoints-{run_name}', type='model')
if save_dir.exists():
    artifact.add_dir(str(save_dir))
if log_dir.exists():
    artifact.add_dir(str(log_dir))

wandb.log_artifact(artifact)
wandb.finish()

print('Uploaded artifact to W&B and finished run.')
print('===== DONE =====')